# From pencil to pixel: assessing Ceramatic 2.0 against manual and laser-aided techniques in archaeological pottery documentation

## Training notebook

This notebook can be used as a template for training the Ceramatic 2.0 for your needs


## Dataset Preparation

### Option 1: Auto-generate labels using existing model
Use the provided script to generate initial labels:
```bash
python generate_yolo_labels.py path/to/images output_dataset --mode auto
```

### Option 2: Manual annotation tools
- **LabelMe**: `pip install labelme` then `labelme`
- **CVAT**: https://www.cvat.ai/
- **Roboflow**: https://roboflow.com/

### Required dataset structure:
```
dataset/
├── train/
│   ├── images/     # .jpg/.png files
│   └── labels/     # .txt files with polygons
├── val/
│   ├── images/
│   └── labels/
└── dataset.yaml    # configuration file
```

### Label format (YOLO polygon):
Each .txt file contains one line per object:
```
0 x1 y1 x2 y2 x3 y3 ... xn yn
```
Where:
- `0` = class id (always 0 for "Profile")
- `x,y` = normalized polygon coordinates (0-1)

### Install the required packages

In [8]:
!pip install ultralytics google

### Import the required packages

In [9]:
from ultralytics import YOLO
from matplotlib import pyplot as plt
from PIL import Image

# Load pre-trained segmentation model
model = YOLO('yolov8m-seg.pt')  # Medium model for segmentation

# For multi-class pottery types (future feature):
# Modify dataset.yaml to include multiple classes:
# names:
#   0: amphora
#   1: olla  
#   2: plate
#   3: bowl

In [10]:
model = YOLO('yolov8m-seg.yaml')  # build a new model from YAML
model = YOLO('yolov8m-seg.pt')  # Transfer the weights from a pretrained model (recommended for training)

### Connect the notebook with Google Drive (usefull if you are going to use Google Colab)

### Dataset configuration file

Create a `dataset.yaml` file with:

```yaml
path: /absolute/path/to/dataset  # dataset root directory
train: train/images  # training images folder
val: val/images      # validation images folder

# Single class for profile extraction:
names:
  0: Profile

# OR multiple classes for typological classification:
# names:
#   0: amphora
#   1: olla
#   2: plate
#   3: bowl
```

**Important**: Labels must be in parallel folders:
- `train/images/` → `train/labels/`
- `val/images/` → `val/labels/`

### Connect to the `.yaml` file

This file contatins the information for training.

```
path: /content/drive/MyDrive/datasets/ollae  # dataset root dir
train: train  # train images (relative to 'path')
val: val # val images (relative to 'path')

names:
  0: Profile
```

The `path` folder contains the images, while `train` and `val` are the folders within tha root `path` folder. `names` indicated the label used in the training. In this case only "profile" must be used.

In [ ]:
# define number of classes based on YAML
import yaml
with open("/content/drive/MyDrive/datasets/ollae_dataset_yolo.yaml", 'r') as stream:
    num_classes = str(yaml.safe_load(stream)['names'])

In [ ]:
# Define a project --> Destination directory for all results
project = "/content/drive/MyDrive/datasets/results"
# Define subdirectory for this specific training
name = "200m_epochs-1024" #note that if you run the training again, it creates a directory: 200_epochs-2

In [ ]:
# Example script to prepare dataset structure
import os
from pathlib import Path
import shutil

def prepare_dataset_structure(source_images, output_path):
    """Create YOLO dataset directory structure"""
    dataset_path = Path(output_path)
    
    # Create directories
    (dataset_path / "train" / "images").mkdir(parents=True, exist_ok=True)
    (dataset_path / "train" / "labels").mkdir(parents=True, exist_ok=True)
    (dataset_path / "val" / "images").mkdir(parents=True, exist_ok=True)
    (dataset_path / "val" / "labels").mkdir(parents=True, exist_ok=True)
    
    print(f"Dataset structure created at: {dataset_path}")
    print("Now add your images and labels to the appropriate folders")
    
# Example: prepare_dataset_structure("raw_images/", "pottery_dataset/")

### Training parameters explained

### Train the model

In [ ]:
# Train the model
results = model.train(data='/content/drive/MyDrive/datasets/ollae_dataset_yolo.yaml',
                      project=project,
                      name=name,
                      epochs=200,
                      patience=100,
                      batch=-1, #8
                      imgsz=1024,
                      device='0',
                      mask_ratio=1,
                      )